# Chapter 11 — Failure Modes, Adversarial Testing and DoE

*Vibes test suites do not cover the behavior space. Design of experiments does.*

## Objective

Cover the catalog of agent failure modes, generate a small design over the factor table, and inspect coverage of the resulting test cases.

In [ ]:
from proofloop.evaluation import (
    ALL_INJECTORS, DEFAULT_FACTORS, FailureMode, balanced_design,
    coverage_report, generate_test_cases, inject,
)

## The failure-mode catalog

Each mode has an injector that mutates a scenario dict. You build adversarial fixtures by composing them.

In [ ]:
for mode, ctor in ALL_INJECTORS.items():
    print(f'  {mode.value:<24} {ctor().description}')

In [ ]:
scenario = {'user_message': 'I have a question.'}
tampered = inject(scenario, FailureMode.PROMPT_INJECTION, FailureMode.WRONG_TOOL)
print(tampered)

## Balanced design over a small factor table

Each factor's levels appear approximately `num_cases / len(levels)` times across the design. Not a true orthogonal design — the goal is coverage of the behavior space, not statistical purity.

In [ ]:
small_factors = {
    'task_complexity': ['single_step', 'multi_step'],
    'user_intent':     ['benign', 'ambiguous', 'adversarial'],
    'evidence':        ['available', 'partial', 'absent'],
}
design = balanced_design(small_factors, num_cases=12, seed=1)
for row in design:
    print(f'  {row}')

print('\ncoverage report:')
for fname, counts in coverage_report(design, small_factors).items():
    print(f'  {fname}: {counts}')

## Generate test cases from the design

`generate_test_cases` wraps the design with a TaskSpec, a user message and an expected behavior. Cases involving `adversarial` intent and `requires_escalation` policy are pre-loaded with the expectation that the agent should escalate.

In [ ]:
cases = generate_test_cases(num_cases=8, seed=42)
for c in cases:
    print(f'  [{c.id}] intent={c.factors["user_intent"]:<11} policy={c.factors["policy"]:<21} '
          f'-> {c.expected_behavior}')

## Space-filling designs: covering interactions, not just levels

Balanced sampling keeps each factor's levels even; a *space-filling* design (Sobol) also covers level **pairs**, which is what attribution needs. `DesignMatrix` is the knowlytix tool the capstone test stand (Chapter 16) uses.

In [ ]:
from knowlytix.harness.graphdoe import DesignMatrix

factors = [
    {'name': 'clarity',         'type': 'categorical',
     'categories': ['clear', 'ambiguous', 'misleading']},
    {'name': 'entity_aliasing', 'type': 'categorical',
     'categories': ['canonical', 'alias', 'typo']},
    {'name': 'reasoning_cue',   'type': 'categorical',
     'categories': ['none', 'cot', 'misleading_cue']},
]
design = DesignMatrix(factors, method='sobol', n_runs=48, seed=7).generate()
for col in ('clarity', 'entity_aliasing', 'reasoning_cue'):
    print(f'  {col:<16} {dict(sorted(design[col].value_counts().items()))}')

## A benchmark is a designed experiment, not a scoreboard

The outcome of each run is binary, so attribution uses **logistic regression** on the factor levels — analysis of deviance, Benjamini–Hochberg correction, odds ratios. Here we plant a clarity effect in a synthetic outcome and confirm `DOEAnalyzer` recovers it (Chapter 16 reads such a table for the real capstone).

In [ ]:
import numpy as np
from knowlytix.harness.graphdoe import DOEAnalyzer

rng = np.random.RandomState(7)
# planted effect: 'misleading' clarity tends to fail
design['correct'] = [(0 if c == 'misleading' else 1) if rng.rand() < 0.85
                     else rng.randint(2) for c in design['clarity']]
analyzer = DOEAnalyzer.from_dataframe(design, factors=[f['name'] for f in factors])
table = analyzer.apply_fdr_correction(analyzer.run_logistic(metric='correct'))
for row in table.to_dict('records'):
    flag = '*' if row['significant_adj'] else ' '
    print(f' {flag} {row["factor"]:<16} p_adj={row["p_value_adj"]:.4f} '
          f'pseudo_r2={row["pseudo_r2"]:.3f}')

## Verifying generated answers against a knowledge-graph ground truth

Binary `correct` works for a label; a generated answer needs a ground truth richer than string matching. Build a GMS from the source document, calibrate its verifiers on the store's own grounded-vs-corrupted triples, then decompose an answer into typed claims and verify each against the graph. Here one fee claim is checked two ways — the committed \$35 reversal cap verifies; the drifted \$80 fails as `FACT_INCORRECT`. Chapter 16 runs the full pipeline against the `search_policy` retriever over a policy-derived GMS.

In [ ]:
from pathlib import Path
import torch
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from knowlytix.harness.testing import FactClaim, TypedVerdict
from knowlytix.harness.testing.judge import GMSJudge

store_path = Path('../data/gms_banking_store')
if not store_path.exists():
    store_path = Path('data/gms_banking_store')
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
store = GMSExpertStore(DocGMSConfig(store_path=str(store_path)), device=dev)
store.load()

# Calibrate the verifiers on the store's own grounded-vs-corrupted triples, then
# wire that operating point into the router (the apply_thresholds step). This is
# in-memory: we do not save(), so the committed calibration.json is untouched.
judge = GMSJudge(store)
judge.calibrate()
judge.apply_thresholds()
print(f'geodesic operating point: {judge._router.graph.plausibility_threshold:.2f}')

# A claim compiler decomposes an answer into typed claims; here we verify one fee
# claim two ways -- the committed $35 reversal cap, and a drifted $80.
for label, tail in [('grounded ($35)', '35.0'), ('drifted ($80)', '80.0')]:
    claim = FactClaim(head='representative', relation='has_max_reversal', tail=tail)
    subs = judge._router.route_all([claim])
    verdict = TypedVerdict.from_sub_verdicts(
        [claim], subs, exact_correct=False, exact_detail='')
    codes = [str(c.code).split('.')[-1] for c in verdict.failures]
    print(f'  {label:14} verified={verdict.n_passed}/{verdict.n_claims}  '
          f'geodesic={subs[0].raw_score:.3f}  codes={codes}')

## Fault injection and release gates: the operational methods

Two methods a regulated deployment needs, both supplied by the knowlytix platform and both applied live to the capstone in Chapter 16: a `FaultProfile` injected through a `ToolGateway` to test recovery, and a risk-tiered `RiskTierProfile` release gate that turns a score into a ship/no-ship decision. Their configuration vocabulary is real and constructable here:

In [ ]:
from knowlytix.harness.testing import FaultProfile
from knowlytix.harness.testing.audit import RiskTierProfile

# a fault profile: error one tool to test the agent's recovery path
fp = FaultProfile(tool_pattern='search_policy', error_rate=1.0,
                  error_message='policy service unavailable')
print('fault:', fp.tool_pattern, 'error_rate=', fp.error_rate)

# escalating release thresholds by how much authority the agent has
for tier in (RiskTierProfile.advisory(), RiskTierProfile.recommendation(),
             RiskTierProfile.action_taking()):
    print(f'  {tier.tier_name:<14} accuracy_threshold={tier.accuracy_threshold}')

## Anti-patterns flagged here

- *Vibes* test suites with hand-picked prompts.
- Single-factor sweeps.
- Adversarial inputs only — never adversarial users.

In [ ]:
# Self-check: the new methodology pieces run.
assert len(design) == 48
assert {'clarity','entity_aliasing','reasoning_cue'} <= {r['factor'] for r in table.to_dict('records')}
assert any(r['factor']=='clarity' and r['significant_adj'] for r in table.to_dict('records'))
assert fp.error_rate == 1.0 and RiskTierProfile.action_taking().accuracy_threshold > RiskTierProfile.advisory().accuracy_threshold
print('self-check OK')
